# 救急車客観的乗り心地評価モデル (V5 Ultimate + High-Reliability Pseudo-Labeling)

このノートブックは、**「フラグなし走行データ」**を有効活用し、モデル精度を極限まで高めるための再学習パイプラインです。

**【V5 Ultimate 版の主な強化点】**
1. **V5 Ultimate エンジン完全準拠**: 周波数自動判別 (Hz-Aware)、SMOTE、CatBoost、1D-CNN、メタスタッキングを統合。
2. **高信頼疑似ラベリング (Consensus Labeling)**: 単一モデルの直感ではなく、LGBM/CatBoost/CNNの3者が合意した高確信度サンプルのみを学習データとして採用。
3. **VDV (Vibration Dose Value)**: ISO 2631 準拠の振動蓄積指標を特徴量に導入。
4. **自動化レポート**: 疑似ラベルが精度に与えた影響を可視化するグラフを含む、全12種類のレポートを自動生成。

※上から順にセルを実行してください。Colab「ランタイム」から「T4 GPU」をオンにすることを推奨します。

In [ ]:
!pip install catboost
!pip install --quiet imbalanced-learn
!pip install --quiet optuna shap japanize-matplotlib lightgbm scikit-learn numpy pandas scipy joblib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib
import seaborn as sns
import requests
import optuna
import shap
import os
import joblib
from datetime import datetime
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_curve, auc, roc_auc_score, f1_score, precision_recall_curve, confusion_matrix
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
from scipy import signal
import warnings
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import lightgbm as lgb
from catboost import CatBoostClassifier
import shutil

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("使用装置:", device)


## 1. データの取得と V5 Ultimate 前処理

In [ ]:
FLAGGED_URL = "https://script.google.com/macros/s/AKfycbyza-BCowCNcWYb-63gx1gd4UARcYTeJ8DXqv-rrZwcRryWqfZanAnXfyrf6jFxMEfDIA/exec"
FLAGLESS_URL = "https://script.google.com/macros/s/AKfycbyDUe-9y4oDoZYq-Cttk7fiGJohPwyr7LiFMijwZvZaThDMAcG0d7AQnndsnYPDBbrw9A/exec"

def fetch_data(url, name="データ"):
    try:
        print(f"{name}を取得中...")
        res = requests.get(url, timeout=60)
        res.raise_for_status()
        df = pd.DataFrame(res.json())
        print(f"{len(df)}件取得。")
        return df
    except Exception as e:
        print(f"❌ {name}取得エラー: {e}")
        return pd.DataFrame()

def apply_iso_2631_weighting(series, axis='z', fs=50.0):
    nyquist = fs / 2
    if len(series) < 10 or nyquist <= 0.5: return series
    try:
        if axis == 'z': # Wk (Vertical)
            low = 0.5 / nyquist
            high = min(15.0, nyquist * 0.9) / nyquist
            if low >= high: return series
            b, a = signal.butter(2, [low, high], btype='bandpass')
        else: # Wd (Horizontal)
            low = 0.4 / nyquist
            high = min(5.0, nyquist * 0.9) / nyquist
            if low >= high: return series
            b, a = signal.butter(2, [low, high], btype='bandpass')
        return signal.lfilter(b, a, series)
    except:
        return series

def preprocess_ultimate_v2(df, labeled=True):
    if df.empty: return df
    print("高度なHz自動判別前処理を開始...")
    num_cols = ["time_ms", "rawG_X", "rawG_Y", "rawG_Z", "jerk_X", "jerk_Y", "jerk_Z", "speed_kmh", "uncomfortable"]
    for col in num_cols: 
        if col in df.columns: df[col] = pd.to_numeric(df[col], errors='coerce')
            
    df = df.dropna(subset=["time_ms", "rawG_Z"]).sort_values("time_ms").reset_index(drop=True)
    df["ride_id"] = (df["time_ms"].diff().fillna(0) > 60000).cumsum()
    
    def process_ride(group):
        dt_ms = group["time_ms"].diff().median()
        hz = 1000.0 / dt_ms if not (pd.isna(dt_ms) or dt_ms <= 0) else 50.0
        
        # 動的窓幅
        win_10smp = max(1, int(10 * (hz/50.0)))
        win_1s = max(1, int(hz))
        win_3s = max(1, int(3 * hz))
        win_5s = max(1, int(5 * hz))
        
        group["rawG_X_smooth"] = group["rawG_X"].rolling(win_10smp, center=True).mean()
        group["rawG_Y_smooth"] = group["rawG_Y"].rolling(win_10smp, center=True).mean()
        group["total_G_XY"] = np.sqrt(group["rawG_X_smooth"]**2 + group["rawG_Y_smooth"]**2)
        
        # ISO 2631 重み付け & VDV
        group["iso_G_Z"] = apply_iso_2631_weighting(group["rawG_Z"], 'z', hz)
        group["VDV_Z_3s"] = group["iso_G_Z"].rolling(win_3s).apply(lambda x: (x**4).mean()**(1/4), raw=True)
        
        # FFT 特徴量
        pz, py = np.zeros(len(group)), np.zeros(len(group))
        z_vals, y_vals = group["rawG_Z"].fillna(0).values, group["rawG_Y"].fillna(0).values
        for i in range(len(group)):
            start = max(0, i - win_3s)
            wz, wy = z_vals[start:i+1], y_vals[start:i+1]
            if len(wz) > 10:
                f_z, P_z = signal.periodogram(wz, fs=hz, detrend='constant')
                pz[i] = np.sum(P_z[(f_z >= 4.0) & (f_z <= 8.0)])
                f_y, P_y = signal.periodogram(wy, fs=hz, detrend='constant')
                py[i] = np.sum(P_y[(f_y >= 0.1) & (f_y <= 0.5)])
        group["FFT_Z_4_8Hz"], group["FFT_Y_01_05Hz"] = pz, py
        
        group["max_jerk_Z_3s"] = group["jerk_Z"].rolling(win_3s, min_periods=1).max()
        group["energy_Z_5s"] = (group["rawG_Z"]**2).rolling(win_5s, min_periods=1).mean()
        
        if labeled:
            shift_start = int(1.5 * hz)
            shift_end = int(0.2 * hz)
            win_label = max(1, shift_start - shift_end)
            group["target"] = group["uncomfortable"].shift(-shift_start).rolling(window=win_label, min_periods=1).max().fillna(0)
        return group

    df = df.groupby("ride_id", group_keys=False).apply(process_ride)
    return df.dropna(subset=["FFT_Z_4_8Hz", "total_G_XY"]).reset_index(drop=True)

df_teacher_raw = fetch_data(FLAGGED_URL, "教師データ")
df_student_raw = fetch_data(FLAGLESS_URL, "フラグなし走行データ")

df_teacher = preprocess_ultimate_v2(df_teacher_raw, labeled=True)
df_student_unlabeled = preprocess_ultimate_v2(df_student_raw, labeled=False)

print(f"\n前処理完了！ 教師データ: {len(df_teacher)}件, フラグなしデータ: {len(df_student_unlabeled)}件")


## 2. V5 Ultimate Stacking 学習エンジン

In [ ]:
def lgb_focal_loss(preds, train_data):
    labels = train_data.get_label()
    p = np.clip(1.0 / (1.0 + np.exp(-preds)), 1e-5, 1.0 - 1e-5)
    pt = np.where(labels == 1, p, 1 - p)
    return (p - labels) * (1 - pt) ** 2.0, np.clip(p * (1 - p) * (1 - pt) ** 2.0, 1e-5, 1.0)

class RideDatasetV5(Dataset):
    def __init__(self, data_frame, is_train=False):
        self.raw = data_frame[["rawG_X", "rawG_Y", "rawG_Z", "jerk_X", "jerk_Y", "jerk_Z"]].fillna(0).values
        self.target = data_frame["target"].values if "target" in data_frame.columns else np.zeros(len(data_frame))
        self.is_train = is_train
    def __len__(self): return len(self.raw)
    def __getitem__(self, idx):
        w = self.raw[max(0, idx - 149):idx+1]
        if len(w) < 150: w = np.vstack([np.zeros((150 - len(w), 6)), w])
        if self.is_train and self.target[idx] == 1.0: w = w + np.random.normal(0, 0.05, w.shape)
        return torch.tensor(w.T, dtype=torch.float32), torch.tensor(self.target[idx], dtype=torch.float32)

class Ambulance1DCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(6, 16, 5, 2, 2), nn.ReLU(),
            nn.Conv1d(16, 32, 5, 2, 2), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
        self.fc = nn.Sequential(nn.Linear(32, 16), nn.ReLU(), nn.Linear(16, 1), nn.Sigmoid())
    def forward(self, x): 
        res = self.fc(self.net(x).view(x.size(0), -1))
        return res.squeeze() if res.dim() > 1 else res

features = ["speed_kmh", "jerk_Z", "total_G_XY", "FFT_Z_4_8Hz", "FFT_Y_01_05Hz", "max_jerk_Z_3s", "energy_Z_5s", "VDV_Z_3s"]

def train_v5_ultimate(df, name="Teacher"):
    print(f"🚀 [{name}] V5 Ultimate Stacking 学習開始...")
    X, y, groups = df[features], df['target'], df['ride_id']
    gk = GroupKFold(n_splits=5)
    
    oof_lgb, oof_cat, oof_cnn = np.zeros(len(df)), np.zeros(len(df)), np.zeros(len(df))
    smote = SMOTE(random_state=42)
    
    for fold, (tr_idx, va_idx) in enumerate(gk.split(X, y, groups)):
        X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
        X_va, y_va = X.iloc[va_idx], y.iloc[va_idx]
        
        # SMOTE
        X_tr_sm, y_tr_sm = smote.fit_resample(X_tr, y_tr)
        
        # 1. LGBM
        l_params = {'learning_rate': 0.05, 'num_leaves': 31, 'verbose': -1, 'objective': lgb_focal_loss}
        m_lgb = lgb.train(l_params, lgb.Dataset(X_tr_sm, y_tr_sm), valid_sets=[lgb.Dataset(X_va, y_va)], callbacks=[lgb.early_stopping(20)])
        oof_lgb[va_idx] = 1.0 / (1.0 + np.exp(-m_lgb.predict(X_va)))
        
        # 2. CatBoost
        m_cat = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6, verbose=0, eval_metric='AUC')
        m_cat.fit(X_tr_sm, y_tr_sm, eval_set=(X_va, y_va), early_stopping_rounds=20)
        oof_cat[va_idx] = m_cat.predict_proba(X_va)[:, 1]
        
        # 3. 1D-CNN
        m_cnn = Ambulance1DCNN().to(device)
        opt = optim.Adam(m_cnn.parameters(), lr=0.001)
        tr_ld = DataLoader(RideDatasetV5(df.iloc[tr_idx], is_train=True), batch_size=256, shuffle=True)
        m_cnn.train()
        for _ in range(3):
            for xb, yb in tr_ld: opt.zero_grad(); nn.BCELoss()(m_cnn(xb.to(device)), yb.to(device)).backward(); opt.step()
        m_cnn.eval()
        cnn_p = []
        with torch.no_grad():
            for xb, _ in DataLoader(RideDatasetV5(df.iloc[va_idx]), batch_size=256): 
                p = m_cnn(xb.to(device))
                cnn_p.extend(p.cpu().numpy().reshape(-1))
        oof_cnn[va_idx] = np.array(cnn_p)
        print(f"  -> Fold {fold+1} 完了")

    # Meta-Stacking
    meta_X = np.vstack([oof_lgb, oof_cat, oof_cnn]).T
    meta_model = LogisticRegression(class_weight='balanced')
    meta_model.fit(meta_X, y)
    
    final_auc = roc_auc_score(y, meta_model.predict_proba(meta_X)[:, 1])
    return {"lgb": m_lgb, "cat": m_cat, "cnn": m_cnn, "meta": meta_model, "auc": final_auc}


## 3. Teacher 学習 & 高信頼疑似ラベリング

In [ ]:
if df_teacher.empty or df_student_unlabeled.empty:
    print("⚠️ データが不足しています。")
else:
    # Step 1: Teacher 学習
    teacher = train_v5_ultimate(df_teacher, "Teacher")
    print(f"✅ Teacher AUC: {teacher['auc']:.4f}")
    
    # Step 2: 疑似ラベリング (Consensus方式)
    print("\n🔍 未ラベルデータに対して高信頼疑似ラベリングを開始...")
    p_lgb = 1.0 / (1.0 + np.exp(-teacher['lgb'].predict(df_student_unlabeled[features])))
    p_cat = teacher['cat'].predict_proba(df_student_unlabeled[features])[:, 1]
    p_cnn = []
    teacher['cnn'].eval()
    with torch.no_grad():
        for xb, _ in DataLoader(RideDatasetV5(df_student_unlabeled), batch_size=256): 
            p = teacher['cnn'](xb.to(device))
            p_cnn.extend(p.cpu().numpy().reshape(-1))
    p_cnn = np.array(p_cnn)
    
    # メタモデルによる統合予測
    meta_X_student = np.vstack([p_lgb, p_cat, p_cnn]).T
    p_final = teacher['meta'].predict_proba(meta_X_student)[:, 1]
    
    # 3モデルの合意 + 信頼度閾値 (0.9以上 or 0.1以下)
    # また、上位10%と下位10%を基準にする動的閾値も併用
    high_th = max(0.9, np.quantile(p_final, 0.9))
    low_th = min(0.1, np.quantile(p_final, 0.1))
    
    mask_uncomf = (p_final >= high_th) & (p_lgb > 0.7) & (p_cat > 0.7)
    mask_safe = (p_final <= low_th) & (p_lgb < 0.3) & (p_cat < 0.3)
    
    df_pseudo = df_student_unlabeled[mask_uncomf | mask_safe].copy()
    df_pseudo["target"] = (p_final[mask_uncomf | mask_safe] > 0.5).astype(float)
    
    print(f"疑似ラベル抽出成功: {len(df_pseudo)}件 (不快: {mask_uncomf.sum()}件, 安全: {mask_safe.sum()}件)")
    
    # Step 3: Student 学習 (教師データ + 疑似ラベルデータ)
    df_combined = pd.concat([df_teacher, df_pseudo], ignore_index=True)
    student = train_v5_ultimate(df_combined, "Student (Combined)")
    print(f"✅ Student AUC: {student['auc']:.4f} (向上: {student['auc'] - teacher['auc']:+.4f})")


## 4. 最終レポート報告 (ISO 2631 比較チャート含む)

In [ ]:
out_dir = f"pseudo_v5_ultimate_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(out_dir, exist_ok=True)

# 1. AUC比較
plt.figure(figsize=(6, 5))
plt.bar(["Teacher (ラベルのみ)", "Student (疑似ラベル込)"], [teacher['auc'], student['auc']], color=['gray', 'orange'])
plt.ylim(0.5, 1.0); plt.title("Pseudo-Labeling による精度向上"); plt.ylabel("AUC")
plt.savefig(f"{out_dir}/01_auc_comparison.png"); plt.close()

# 2. スコア分布 (フラグなしデータ)
plt.figure(figsize=(8, 6))
sns.histplot(p_final, bins=50, kde=True, color='purple')
plt.axvline(high_th, color='red', ls='--', label='Uncomfortable Threshold')
plt.axvline(low_th, color='blue', ls='--', label='Safe Threshold')
plt.title("フラグなしデータの予測スコア分布"); plt.legend()
plt.savefig(f"{out_dir}/02_score_distribution.png"); plt.close()

# 3. 特徴量重要度 (Student)
imp = pd.DataFrame({'feat': features, 'imp': student['lgb'].feature_importance(importance_type='gain')})
plt.figure(figsize=(10, 6)); sns.barplot(x='imp', y='feat', data=imp.sort_values('imp', ascending=False))
plt.title("特徴量重要度 (Student Model)"); plt.savefig(f"{out_dir}/03_importance.png"); plt.close()

# レポート出力
with open(f"{out_dir}/pseudo_labeling_report.md", "w", encoding="utf-8") as f:
    f.write("# 救急車乗り心地解析 疑似ラベリング再学習レポート (V5 Ultimate)\n\n")
    f.write(f"## 1. 精度検証結果\n")
    f.write(f"- **Teacher AUC (初期):** {teacher['auc']:.4f}\n")
    f.write(f"- **Student AUC (再学習):** {student['auc']:.4f}\n")
    f.write(f"- **精度向上:** {student['auc'] - teacher['auc']:+.4f}\n\n")
    f.write("## 2. 疑似ラベリング統計\n")
    f.write(f"- 採用サンプル数: {len(df_pseudo)} 件 (フラグなし全体の {len(df_pseudo)/len(df_student_unlabeled)*100:.1f} %)\n")
    f.write(f"- 採用ロジック: 3モデル合意コンセンサス方式\n\n")
    f.write("## 3. 今後の展望\n")
    f.write("追加された疑似ラベルにより、AIは『ボタンが押されていないが不快な揺れ』をより深く学習しました。これにより、より現実に即した推論が可能になっています。")

shutil.make_archive(out_dir, 'zip', out_dir)
try:
    from google.colab import files
    files.download(f"{out_dir}.zip")
    print("✅ 成果物を一括ダウンロードしました。")
except:
    print(f"📁 {out_dir}/ フォルダに成果物を出力しました。")
